# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第2章 研究課題と仮説の設計 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch02_hypothesis_descent.R`

対象章: 第2章「研究課題と仮説の設計」

「理論的問いから検証可能な仮説へ」節のフロー図。理論的な問いが、
予測・操作的定義を経て測定可能な変数へと降下していく段階を、
本文の具体例(モーラの実在性)に沿って図示する概念図。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

stages <- tibble(
  step  = 1:4,
  label = c(
    "理論的な問い\n「モーラは心理的に実在するか」",
    "具体的な予測\n「モーラ境界での判断は、\nより速く・より一貫しているはずだ」",
    "操作的定義\n「速さ」→反応時間\n「一貫性」→正答率のばらつき",
    "測定可能な変数\n反応時間(ms)、正答率のSD"
  ),
  y = rev(1:4)
)

p <- ggplot(stages) +
  geom_rect(aes(xmin = 0, xmax = 10, ymin = y - 0.35, ymax = y + 0.35),
            fill = "#34518A", alpha = 0.12, colour = "#34518A") +
  geom_text(aes(x = 5, y = y, label = label), size = 3.2, colour = "#1B2333", lineheight = 0.9) +
  geom_segment(data = stages |> filter(step < 4),
               aes(x = 5, xend = 5, y = y - 0.38, yend = y - 0.65),
               arrow = arrow(length = unit(0.18, "cm"), type = "closed"), colour = "#A8681F") +
  scale_x_continuous(limits = c(0, 10)) +
  scale_y_continuous(limits = c(0.5, 4.5)) +
  theme_void() +
  labs(caption = "概念図。scripts/R/ch02_hypothesis_descent.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.8), colour = "grey35"))

save_fig(p, "fig02-01-hypothesis-descent", chapter = "ch02", width = 5.5, height = 6)
cat("完了。図はassets/figures/ch02/に保存した。\n")


## `ch02_within_between_design.R`

対象章: 第2章「研究課題と仮説の設計」

「実験デザインの基本用語」節の作例。同一の効果量を、参加者内デザイン
(個人差を対応のあるt検定で除去)と参加者間デザイン(個人差を除去せず
対応のないt検定)のそれぞれで検出する経験的検定力を、実際に
シミュレーションで比較する。架空のデータであり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_participants <- 20
true_effect    <- 10     # 条件差(理論的に意味のある最小の効果量)
individual_sd  <- 25     # 参加者間の個人差(ベースラインの違い)
residual_sd    <- 12     # 試行ごとの残差誤差
n_sim <- 500

simulate_within <- function() {
  baseline <- rnorm(n_participants, 200, individual_sd)
  cond_a <- baseline + rnorm(n_participants, 0, residual_sd)
  cond_b <- baseline + true_effect + rnorm(n_participants, 0, residual_sd)
  t.test(cond_b, cond_a, paired = TRUE)$p.value < .05
}

simulate_between <- function() {
  baseline_a <- rnorm(n_participants, 200, individual_sd)
  baseline_b <- rnorm(n_participants, 200, individual_sd)
  cond_a <- baseline_a + rnorm(n_participants, 0, residual_sd)
  cond_b <- baseline_b + true_effect + rnorm(n_participants, 0, residual_sd)
  t.test(cond_b, cond_a, paired = FALSE)$p.value < .05
}

power_within  <- mean(replicate(n_sim, simulate_within()))
power_between <- mean(replicate(n_sim, simulate_between()))

cat("参加者内デザイン(個人差を対応のあるt検定で除去): 検定力 =",
    round(power_within * 100, 1), "%\n")
cat("参加者間デザイン(個人差を除去せず): 検定力 =",
    round(power_between * 100, 1), "%\n")


## `ch02_pilot_inflation.R`

対象章: 第2章「研究課題と仮説の設計」

「検定力とサンプルサイズ」節の作例。小規模な予備実験(パイロット
スタディ)で「有意だった」効果量だけを採用してサンプルサイズを見積もると、
その効果量が真の効果量を系統的に過大評価してしまう現象(いわゆる
winner's curse)を、実際にシミュレーションで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

true_effect   <- 8
sd_pop        <- 20
n_pilot       <- 12
n_sim         <- 2000

pilot_once <- function() {
  a <- rnorm(n_pilot, 100, sd_pop)
  b <- rnorm(n_pilot, 100 + true_effect, sd_pop)
  test <- t.test(b, a)
  list(sig = test$p.value < .05, estimate = mean(b) - mean(a))
}

results <- map(1:n_sim, ~ pilot_once())
sig_results <- keep(results, ~ .x$sig)

mean_all_estimates <- mean(map_dbl(results, "estimate"))
mean_sig_estimates <- mean(map_dbl(sig_results, "estimate"))

cat("真の効果量:", true_effect, "\n")
cat("全", n_sim, "回の平均推定値(有意/非有意を問わず):", round(mean_all_estimates, 1), "\n")
cat("有意だった", length(sig_results), "回(", round(length(sig_results) / n_sim * 100, 1),
    "%)だけに限った平均推定値:", round(mean_sig_estimates, 1), "\n")
cat("過大評価の倍率:", round(mean_sig_estimates / true_effect, 2), "倍\n")


## `ch02_power_simulation.R`

対象章: 第2章「研究課題と仮説の設計」

「検定力とサンプルサイズ」節の作例。Westfall, Kenny, and Judd (2014)
が示した「話者数だけを増やしても、項目数が少なければ検定力は頭打ちになる」
という主張を、架空のデータに対する実際のシミュレーションで確認する。
実測データではない。

設計: 話者数(20名/100名)×項目数(5語/20語)の2×2で、各組み合わせにつき
5000回のシミュレーションを繰り返し、条件差を検出できた割合(検定力)を
経験的に求める。真の効果量は、話者・項目のランダム切片によるばらつきに
対して中程度になるよう設定する。反復数は、報告する検定力の値どうしを
数ポイント単位で比較できる程度までMonte Carlo標準誤差を抑えるために
5000回とした(真の検定力が15%付近ならSEは約0.5ポイント、45%付近でも
約0.7ポイントで、150回時点のSE(4ポイント前後)より1桁小さい)。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

true_effect     <- 14    # 条件差の真の値(ms、小さめに設定し天井効果を避ける)
speaker_sd      <- 20    # 話者間のランダム切片のSD
item_sd         <- 25    # 項目間のランダム切片のSD(項目数が少ないと効いてくる)
residual_sd     <- 25    # 試行内の残差SD
n_reps_per_cell <- 3     # 話者×項目×条件あたりの反復数
n_sim           <- 5000  # 各組み合わせでのシミュレーション回数

simulate_one <- function(n_speakers, n_items) {
  # 各項目は条件A・条件Bのどちらか一方にのみ属する(語彙リストA/Bのように、
  # 刺激そのものが条件を担う設計)。これにより、項目数が少ないと項目間の
  # ばらつきが条件効果の推定と直接交絡する、Clark (1973)の
  # "language-as-fixed-effect-fallacy"型の状況を再現できる。
  speakers <- tibble(speaker = sprintf("s%02d", 1:n_speakers), spk_re = rnorm(n_speakers, 0, speaker_sd))
  items <- tibble(
    item      = sprintf("i%02d", 1:(2 * n_items)),
    condition = rep(c("A", "B"), each = n_items),
    item_re   = rnorm(2 * n_items, 0, item_sd)
  )
  d <- speakers |>
    tidyr::crossing(items) |>
    tidyr::crossing(rep = 1:n_reps_per_cell) |>
    mutate(
      cond_c = if_else(condition == "B", 0.5, -0.5),
      y = 200 + true_effect * cond_c + spk_re + item_re + rnorm(n(), 0, residual_sd)
    )
  m <- tryCatch(
    lmerTest::lmer(y ~ condition + (1 | speaker) + (1 | item), data = d,
                    control = lmerControl(optimizer = "bobyqa")),
    error = function(e) NULL, warning = function(w) NULL
  )
  if (is.null(m)) return(NA)
  p <- tryCatch(summary(m)$coefficients["conditionB", "Pr(>|t|)"], error = function(e) NA)
  p
}

conditions_grid <- tibble::tribble(
  ~n_speakers, ~n_items,
  20,          5,
  100,         5,
  20,          20,
  100,         20
)

set.seed(20260906)
results <- conditions_grid |>
  rowwise() |>
  mutate(power = {
    ps <- replicate(n_sim, simulate_one(n_speakers, n_items))
    mean(ps < .05, na.rm = TRUE)
  }) |>
  ungroup()

print(results)
write_csv(results, here::here("data", "processed", "ch02_power_results.csv"))
cat("完了。data/processed/ch02_power_results.csv に保存した。\n")
